Lecture simple d'un fichier texte en Python

In [ ]:
import os
with open('../data/text_samples/intro_rag.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    print(text)

In [ ]:
from langchain_core.documents import Document
doc = Document(page_content=text, metadata={"source": "../data/text_samples/intro_rag.txt", 
                                            "format": "txt"})

In [ ]:
print(doc)

In [ ]:
print("contenu", doc.page_content[:80])
print("metadata", doc.metadata)

Utilisation de textLoaders pour lire un fichier texte et créer un document LangChain

In [ ]:
from langchain_community.document_loaders import TextLoader

Créaction du chargeur de texte 

In [ ]:
loader = TextLoader('../data/text_samples/intro_rag.txt', encoding='utf-8')

In [ ]:
docs = loader.load()

In [ ]:
type(docs), len(docs)

In [ ]:
docs[0].metadata

Création du second fichier texte

In [ ]:
sample_text = """
Les fichiers texte sont la forme la plus simple de données à ingérer.
Mais dans un projet réel, tu auras souvent des PDF, des CSV, ou des documents Word.
L'idée est de tout standardiser sous forme de Documents pour faciliter le traitement en aval.
"""

with open('../data/text_samples/ingestion_2.txt', 'w', encoding='utf-8') as f:
    f.write(sample_text)
    print("nouveau fichier texte créé : ../data/text_samples/ingestion_2.txt")

Démonstation pour plusieurs fichiers texte

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
multi_loader = DirectoryLoader('../data/text_samples', glob="*.txt", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})  
documents = multi_loader.load()

print(f"Nombre de documents chargés : {len(documents)}")
for d in documents:
    print(d.metadata)

Ajout de metadata personnalisée pour chaque document

In [ ]:
for doc in documents:
    if "intro" in doc.metadata["source"]:
        doc.metadata["theme"] = "introduction"
    elif "ingestion" in doc.metadata["source"]:
        doc.metadata["theme"] = "directoryLoader"

    doc.metadata["author"] = "Lady"
    doc.metadata["format"] = "text"
    doc.metadata["version"] = "v1.0"

for doc in documents:
    print(doc.metadata)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../data/text_samples/texte_exemple_rag.txt', encoding='utf-8')
documents = loader.load()

print(documents[0].page_content[:300])

In [ ]:
def clean_text(text :str) -> str:
    import re
    text = re.sub(r'\s+', ' ', text)  # Remplace les espaces multiples par un seul espace
    text = text.strip()  # Supprime les espaces au début et à la fin du texte
    return text

In [ ]:
def clean_documents(documents):
    cleaned_documents = []
    for doc in documents:
        if not doc.page_content:
            doc.page_content = ""
        else:
            doc.page_content = clean_text(doc.page_content)
        cleaned_documents.append(doc)
    return cleaned_documents

In [ ]:
cleaned_docs = clean_documents(documents)

print("Avant nettoyage :", documents[0].page_content[:300])
print("Après nettoyage :", cleaned_docs[0].page_content[:300])

Découper les documents (Document Splitters)

Ce qui fait un bon chunk\
    Un bon chunk :
    - Il doit contenir une seule idée principale,
    - Avec un minimum de contexte pour éviter les erreurs d’interprétation,
    - Pas trop long (risque de dépasser le contexte du modèle),
    - Pas trop court (le modèle manque d'information)

Le splitter utilisé dans 95% des projets : **RecursiveCharacterTextSplitter**

1. doubles sauts de ligne

2. sauts de ligne

3. points

4. espaces

5. seulement en dernier recours : caractère par caractère

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # taille idéale d’un chunk en caractères
    chunk_overlap=150,    # chevauchement entre les chunks
    separators=["\n\n", "\n", ".", " ", ""]  # ordre logique de découpe
)

In [ ]:
chunks = splitter.split_documents(cleaned_docs)

In [ ]:
# Inspecter les chunks,
print(f"Nombre total de chunks : {len(chunks)}")
print(f" chunk1: {chunks[0].page_content}")
print(f" chunk2: {chunks[1].page_content}")
print(f" chunk3: {chunks[2].page_content}")